# Inclusive-jet data distributions

Beam-orientation comparisons for configurable reconstructed-jet selections. For the unflipped-Lab, flipped-Lab, CM, and raw-pT/unflipped-Lab views, plot common-scale Pb-going, p-going, and combined 2D maps, normalized pT spectra with ratios to combined, and eta projections with ratios to combined.

All four triggers use the shared data-output resolver, which maps each configured selection to the combined, Pb-going, and p-going merged filenames.


<!-- detailed-workflow-guide -->

### Detailed workflow and data interpretation

Data histograms contain selected event or jet yields for explicit trigger, direction, and selection inputs. A projected bin content is the stored weighted count $N_i=\sum w_e$, with uncertainty inherited from the ROOT histogram. Trigger samples are not combined implicitly: each configured interval uses the documented trigger or stitching rule.

Unit-area normalization tests shape and removes luminosity and trigger-yield information. Density normalization additionally divides by bin width, $p_i=N_i/(\Delta x_i\sum_jN_j)$. Logarithmic display changes only rendering. Empty or prescaled regions should be diagnosed from the source trigger rather than interpreted as physical suppression.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
from dataclasses import replace
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.data_distributions import (
    draw_corrected_raw_eta_comparisons,
    draw_orientation_comparisons,
)
from hist_analysis.python.histogram_io import resolve_data_file
from hist_analysis.python.root_style import DEFAULT_PLOT_STYLE


In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
SELECTION = 'jetId'  # jetId, trkMax, or noSel
if SELECTION not in {'jetId', 'trkMax', 'noSel'}: raise ValueError(f'Unsupported SELECTION={SELECTION!r}')
OUTPUT_DIR = Path(os.environ.get('DATA_JET_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis/output/data_jet_distributions')) / SELECTION
TRIGGERS = ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
DIRECTION_FILES = {trigger: {
    label: resolve_data_file(DATA_DIR, trigger, direction, SELECTION)
    for label, direction in (
        ('Pb-going', 'Pbgoing'), ('p-going', 'pgoing'), ('combined', 'combined'),
    )
} for trigger in TRIGGERS}
FRAMES = {
    'lab_unflipped': ('Lab unflipped', 'hRecoInclusiveJetPtEtaLabUnflipped', '#eta_{Lab,unflipped}^{jet}'),
    'raw_lab_unflipped': ('Raw pT, Lab unflipped', 'hRecoInclusiveJetRawPtEtaLabUnflipped', '#eta_{Lab,unflipped}^{jet}', 'p_{T}^{raw,jet}'),
    'lab': ('Lab flipped', 'hRecoInclusiveJetPtEtaLab', '#eta_{Lab}^{jet}'),
    'cm': ('CM', 'hRecoInclusiveJetPtEtaCM', '#eta_{CM}^{jet}'),
}
JET_PT_BINS = {
    'MinimumBias': [(40, 60), (60, 80), (80, 100), (100, 120), (120, 140)],
    'Jet60': [(60, 80), (80, 100), (100, 120), (120, 140), (140, 180)],
    'Jet80': [(80, 100), (100, 120), (120, 140), (140, 180), (200, 300)],
    'Jet100': [(100, 120), (120, 140), (140, 180), (200, 300), (300, 500)],
}
PT_ETA_RANGE = (-2.5, 2.5)  # half-open eta range for pT projections
ETA_DISPLAY_RANGE = (-3.5, 3.5)  # (eta_low, eta_high), or None for the full axis
ETA_Y_RANGE = (0., 0.015)  # (y_low, y_high), or None for automatic scaling
CORRECTED_RAW_RATIO_RANGE = (1., 3.5)
CORRECTED_RAW_DIRECTION_RATIO_RANGE = (0.75, 1.25)
PT_ORIENTATION_NORMALIZATION_RANGE = (110.0, 130.0)
REBIN_PT = 1; REBIN_ETA = 1; PT_DISPLAY_RANGE = (40.0, 500.0)
ORIENTATION_RATIO_RANGE = (0.75, 1.25)
SAVE_PNG = False; DRAW_GRID = True
PLOT_STYLE = replace(DEFAULT_PLOT_STYLE, annotation_text_size=0.028, legend_text_size=0.028)
missing = [str(path) for files in DIRECTION_FILES.values() for path in files.values() if not path.exists()]
if missing: raise FileNotFoundError('Missing ROOT files:\n' + '\n'.join(sorted(set(missing))))


## Pb-going and p-going overlays and ratios to combined

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
mb_orientation_results = draw_orientation_comparisons(
    'MinimumBias', DIRECTION_FILES['MinimumBias'], FRAMES, JET_PT_BINS['MinimumBias'],
    jet_kind='jet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=PT_ETA_RANGE,
    eta_display_range=ETA_DISPLAY_RANGE, eta_y_range=ETA_Y_RANGE,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE,
)
mb_corrected_raw_results = draw_corrected_raw_eta_comparisons(
    'MinimumBias', DIRECTION_FILES['MinimumBias'], JET_PT_BINS['MinimumBias'],
    corrected_key=FRAMES['lab_unflipped'][1], raw_key=FRAMES['raw_lab_unflipped'][1],
    output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, eta_display_range=ETA_DISPLAY_RANGE,
    ratio_range=CORRECTED_RAW_RATIO_RANGE,
    direction_ratio_range=CORRECTED_RAW_DIRECTION_RATIO_RANGE, selection=SELECTION,
    save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet60_orientation_results = draw_orientation_comparisons(
    'Jet60', DIRECTION_FILES['Jet60'], FRAMES, JET_PT_BINS['Jet60'],
    jet_kind='jet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=PT_ETA_RANGE,
    eta_display_range=ETA_DISPLAY_RANGE, eta_y_range=ETA_Y_RANGE,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE,
)
jet60_corrected_raw_results = draw_corrected_raw_eta_comparisons(
    'Jet60', DIRECTION_FILES['Jet60'], JET_PT_BINS['Jet60'],
    corrected_key=FRAMES['lab_unflipped'][1], raw_key=FRAMES['raw_lab_unflipped'][1],
    output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, eta_display_range=ETA_DISPLAY_RANGE,
    ratio_range=CORRECTED_RAW_RATIO_RANGE,
    direction_ratio_range=CORRECTED_RAW_DIRECTION_RATIO_RANGE, selection=SELECTION,
    save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet80_orientation_results = draw_orientation_comparisons(
    'Jet80', DIRECTION_FILES['Jet80'], FRAMES, JET_PT_BINS['Jet80'],
    jet_kind='jet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=PT_ETA_RANGE,
    eta_display_range=ETA_DISPLAY_RANGE, eta_y_range=ETA_Y_RANGE,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE,
)
jet80_corrected_raw_results = draw_corrected_raw_eta_comparisons(
    'Jet80', DIRECTION_FILES['Jet80'], JET_PT_BINS['Jet80'],
    corrected_key=FRAMES['lab_unflipped'][1], raw_key=FRAMES['raw_lab_unflipped'][1],
    output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, eta_display_range=ETA_DISPLAY_RANGE,
    ratio_range=CORRECTED_RAW_RATIO_RANGE,
    direction_ratio_range=CORRECTED_RAW_DIRECTION_RATIO_RANGE, selection=SELECTION,
    save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet100_orientation_results = draw_orientation_comparisons(
    'Jet100', DIRECTION_FILES['Jet100'], FRAMES, JET_PT_BINS['Jet100'],
    jet_kind='jet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=PT_ETA_RANGE,
    eta_display_range=ETA_DISPLAY_RANGE, eta_y_range=ETA_Y_RANGE,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE,
)
jet100_corrected_raw_results = draw_corrected_raw_eta_comparisons(
    'Jet100', DIRECTION_FILES['Jet100'], JET_PT_BINS['Jet100'],
    corrected_key=FRAMES['lab_unflipped'][1], raw_key=FRAMES['raw_lab_unflipped'][1],
    output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, eta_display_range=ETA_DISPLAY_RANGE,
    ratio_range=CORRECTED_RAW_RATIO_RANGE,
    direction_ratio_range=CORRECTED_RAW_DIRECTION_RATIO_RANGE, selection=SELECTION,
    save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)